[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/solutions_03_01_exercise_guided_2.ipynb)

In [ ]:
# --- Course setup (uncomment when running on Colab) ---
#!git clone https://github.com/tunnel-ai/way.git
#import sys; sys.path.insert(0, "/content/way/src")

# Module 3 — Classification (Solution 2 — Tuned Logit)

**Variant of** `solutions_03_01_exercise_guided.ipynb`.

Same data, same engineered features, but two upgrades on the modeling side:

1. **`PowerTransformer(method="yeo-johnson")`** in place of `StandardScaler` on the numeric branch. Transaction amount and velocity counts are heavy-tailed; `StandardScaler` only centers/scales, while Yeo-Johnson learns a per-feature power transform that pulls the tails in *and* standardizes. Often beats hand-rolled `log1p` because the transform is fit per feature.
2. **`GridSearchCV`** over `C`, `penalty` (L1 vs L2), and `class_weight`. The default `C=1.0` and the blunt `class_weight="balanced"` are rarely optimal. We score on `average_precision` (PR-AUC) since that's what we care about at a 4% base rate.

We compare three models at the end: original logit, engineered logit, tuned + power-transformed logit.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
)

RANDOM_STATE = 1955
np.random.seed(RANDOM_STATE)

In [ ]:
from core.generators.transaction_risk_dgp import generate_transaction_risk_dataset

df = generate_transaction_risk_dataset(seed=RANDOM_STATE)

print(df.shape)
df.head()

## 1) Split (same leakage hygiene as solution 1)

In [ ]:
TARGET = "is_fraud"

DROP_COLS = [
    TARGET,
    "transaction_loss_amount",
    "chargeback_flag",
    "manual_review_score",
    "fraud_probability_latent",
    "transaction_id",
    "account_id",
    "merchant_description",
    "merchant_name",
]

X = df.drop(columns=DROP_COLS).copy()
X["merchant_id"] = X["merchant_id"].astype(str)
y = df[TARGET].astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print("Train fraud rate:", y_train.mean())
print("X_train shape:  ", X_train.shape)

## 2) Engineer features (same as solution 1)

We keep the four engineered columns from the original solution. Note: `log_amount` and `log_velocity_24h` are still useful even alongside `PowerTransformer` because they encode the analyst's hypothesis directly. Yeo-Johnson on the *raw* `transaction_amount` would learn a similar shape, but redundancy here is cheap — the L1 penalty in the grid will zero out whichever copy isn't pulling its weight.

In [ ]:
def engineer_features(X):
    X = X.copy()
    X["foreign_new_device_online"] = (
        X["is_foreign_transaction"]
        * X["is_new_device"]
        * (X["payment_channel"] == "online").astype(int)
    )
    night_hours = {0, 1, 2, 3, 4, 5, 21, 22, 23}
    X["is_night"] = X["hour_of_day"].isin(night_hours).astype(int)
    X["log_amount"] = np.log1p(X["transaction_amount"])
    X["log_velocity_24h"] = np.log1p(X["transactions_last_24h"])
    return X

X_train_eng = engineer_features(X_train)
X_valid_eng = engineer_features(X_valid)

high_card_cols = ["merchant_id"]
categorical_cols = X_train_eng.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
low_card_cols = [c for c in categorical_cols if c not in high_card_cols]
numeric_cols = [c for c in X_train_eng.columns if c not in categorical_cols]

print("numeric cols:", len(numeric_cols))
print("low-card cat cols:", low_card_cols)
print("high-card cat cols:", high_card_cols)

## 3) Preprocessor — PowerTransformer on numerics

Only the numeric branch changes. `PowerTransformer(method="yeo-johnson", standardize=True)` handles imputed values and produces zero-mean, unit-variance output, so the imputer is still upstream and no separate `StandardScaler` is needed.

In [ ]:
numeric_transformer_pt = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("power", PowerTransformer(method="yeo-johnson", standardize=True)),
])
low_card_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
high_card_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50)),
])

preprocess_pt = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_pt, numeric_cols),
        ("cat_low", low_card_transformer, low_card_cols),
        ("cat_high", high_card_transformer, high_card_cols),
    ],
    remainder="drop",
)

## 4) Grid search

We sweep:

- `C` ∈ {0.01, 0.1, 1.0, 10.0} — strength of regularization (smaller = stronger)
- `penalty` ∈ {L1, L2} — L1 sparsifies the high-cardinality merchant one-hots
- `class_weight` ∈ {`balanced`, custom 1:25, custom 1:50}

Solver is `saga` (only sklearn solver that handles both L1 and L2 with `class_weight`). Scoring is **`average_precision`** — PR-AUC, the metric we actually care about. 3-fold stratified CV keeps runtime reasonable on this dataset.

**Grid size:** 4 × 2 × 3 = 24 configs × 3 folds = 72 fits. Expect this cell to take a few minutes.

In [ ]:
logit_pipe = Pipeline(steps=[
    ("preprocess", preprocess_pt),
    ("model", LogisticRegression(
        solver="saga",
        max_iter=4000,
        random_state=RANDOM_STATE,
    )),
])

param_grid = {
    "model__C":           [0.01, 0.1, 1.0, 10.0],
    "model__penalty":     ["l1", "l2"],
    "model__class_weight": ["balanced", {0: 1, 1: 25}, {0: 1, 1: 50}],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=logit_pipe,
    param_grid=param_grid,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

grid.fit(X_train_eng, y_train)

print("Best CV PR-AUC:", round(grid.best_score_, 4))
print("Best params:   ", grid.best_params_)

### Inspect the top configs

Sometimes the "best" config is barely better than several alternatives — useful to know whether the win is robust or noise.

In [ ]:
results = pd.DataFrame(grid.cv_results_)
cols = ["param_model__C", "param_model__penalty", "param_model__class_weight",
        "mean_test_score", "std_test_score", "rank_test_score"]
results[cols].sort_values("rank_test_score").head(10)

## 5) Evaluate on the held-out validation set

In [ ]:
y_proba_tuned = grid.predict_proba(X_valid_eng)[:, 1]

print("Tuned ROC-AUC:", round(roc_auc_score(y_valid, y_proba_tuned), 4))
print("Tuned PR-AUC: ", round(average_precision_score(y_valid, y_proba_tuned), 4))

## 6) Compare against solution 1

Refit the two baselines from solution 1 (raw and engineered, both `StandardScaler` + `class_weight="balanced"` + default `C`) so we can put all three on the same row.

In [ ]:
# Baseline preprocessor (StandardScaler) — same as solution 1
numeric_transformer_ss = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Raw features — only the original numeric columns (drop engineered)
raw_numeric_cols = [c for c in numeric_cols if c not in
                    ["foreign_new_device_online", "is_night", "log_amount", "log_velocity_24h"]]

preprocess_raw = ColumnTransformer(transformers=[
    ("num", numeric_transformer_ss, raw_numeric_cols),
    ("cat_low", low_card_transformer, low_card_cols),
    ("cat_high", high_card_transformer, high_card_cols),
], remainder="drop")

preprocess_eng_ss = ColumnTransformer(transformers=[
    ("num", numeric_transformer_ss, numeric_cols),
    ("cat_low", low_card_transformer, low_card_cols),
    ("cat_high", high_card_transformer, high_card_cols),
], remainder="drop")

def fit_baseline(preprocess, X_tr, X_va):
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(
            max_iter=2000, class_weight="balanced",
            solver="liblinear", random_state=RANDOM_STATE,
        )),
    ])
    pipe.fit(X_tr, y_train)
    return pipe.predict_proba(X_va)[:, 1]

y_proba_raw = fit_baseline(preprocess_raw, X_train, X_valid)
y_proba_eng = fit_baseline(preprocess_eng_ss, X_train_eng, X_valid_eng)

rows = []
for name, proba in [
    ("1. Raw + StandardScaler",         y_proba_raw),
    ("2. Engineered + StandardScaler",  y_proba_eng),
    ("3. Engineered + PowerTransformer + GridSearch", y_proba_tuned),
]:
    rows.append({
        "model":   name,
        "ROC_AUC": roc_auc_score(y_valid, proba),
        "PR_AUC":  average_precision_score(y_valid, proba),
    })
pd.DataFrame(rows).round(4)

## 7) Re-pick the operating threshold on the tuned model

Threshold from solution 1 was tuned to a different probability distribution. With `class_weight` and `C` both retuned, the tuned model's probabilities will be calibrated differently — re-run the cost minimization.

In [ ]:
C_FN = 50
C_FP = 1

precision, recall, thresholds = precision_recall_curve(y_valid, y_proba_tuned)

costs = []
for t in thresholds:
    y_hat = (y_proba_tuned >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, y_hat).ravel()
    costs.append(C_FN * fn + C_FP * fp)
costs = np.array(costs)

best_idx = costs.argmin()
best_t = thresholds[best_idx]
y_pred_best = (y_proba_tuned >= best_t).astype(int)

print("Best threshold:    ", round(float(best_t), 4))
print("Min expected cost: ", int(costs[best_idx]))
print(confusion_matrix(y_valid, y_pred_best))
print(classification_report(y_valid, y_pred_best, digits=4))

## 8) Discussion

**What to look for in the comparison:**

- If the tuned PR-AUC beats the engineered baseline by a meaningful margin, the win is some combination of (a) Yeo-Johnson reshaping the heavy-tailed numerics more usefully than `log1p` + standardize, and (b) `C`/`class_weight` finding a regularization sweet spot the defaults missed.
- If `penalty=l1` shows up in the best params, the high-cardinality merchant one-hots had columns the model didn't need — L1 zeroed them out and the remaining signal got cleaner.
- If the win is small (e.g. <0.005 PR-AUC), the honest read is that logit's ceiling on this dataset is roughly where solution 1 left it. The next move would be a non-linear model (gradient boosting), not more logit tuning.

**Caveat on the cost-minimizing threshold:** changing `class_weight` shifts the absolute probability scale of `predict_proba`. The tuned threshold is not directly comparable to solution 1's threshold — only the *resulting* confusion matrix and total cost are. If you need probabilities to remain interpretable across model versions (e.g. shared with downstream systems), wrap the final model in `CalibratedClassifierCV(method="isotonic")`.